# 3D Segmentation Backbones → Glaucoma Classifier — Experiment Sweep

Explore **well-known 3D medical segmentation architectures** as *encoders* (decoder dropped)
+ a classification head, on OCT volumes streamed from HF (`harvardairobotics/Harvard-GF`).

| Backbone | Type | Origin |
|---|---|---|
| `dynunet` | conv (residual) | MONAI DynUNet = nnU-Net v1 / ResEnc-style |
| `vnet` | conv | V-Net (3D volumetric) |
| `segresnet` | conv + residual blocks | SegResNet (3D MRI segmentation) |
| `unetr` | ViT encoder | UNETR (transformer) |
| `swinunetr` | Swin-transformer encoder | SwinUNETR |
| `nnunetv2-plain` | conv encoder | **official nnU-Net v2** (`dynamic_network_architectures`) |
| `nnunetv2-res` | residual encoder | **official nnU-Net v2** (ResEnc) |
| `unetenc-32` | MONAI UNet encoder | baseline (our earlier UNetEncoder3D) |

Each runs the ENCODER only and appends `AdaptiveAvgPool3d → Dropout → Linear(2)`.

**Resolution:** this notebook defaults to **128³** so every backbone works (SwinUNETR needs input
divisible by 32; `RESOLUTION` is a top-of-cell constant — set 192 or 200 for conv-only backbones).
A100 80GB, bf16 AMP, batch auto-probed (auto-drops to 1 on OOM), grad-accum.


In [ ]:
!pip -q install monai umap-learn
!pip -q install nnunetv2
!pip -q install hf-transfer huggingface_hub datasets


In [ ]:
# HF token (Colab Secrets -> HF_TOKEN -> hf_xxx) + Drive for saving results
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")


In [ ]:
# ====== keep the Colab session alive during long runs ======
# Re-clicks Colab's "connect" button every 60s so an idle browser tab does not
# kill the runtime. Keep this tab open and unfocused is fine — but don't close it.
from google.colab import output

JS = """
setInterval(function(){
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.click();
}, 60000);
"""
try:
    output.eval_js(JS)
    print("[keepalive] armed — runtime will auto-reconnect while this tab stays open.")
except Exception as e:
    print("[keepalive] not available:", e)


In [ ]:
import os, io, json, time, zipfile
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from monai.networks import nets
from monai.networks.blocks import UnetBasicBlock
from monai.transforms import (Compose, RandFlip, RandRotate, RandScaleIntensity,
                              RandShiftIntensity, RandGaussianNoise)

# ---- device / AMP ----
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True
_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if _bf16 else torch.float16
SEED = 42


def to_device_normalize(x, y):
    x = x.to(device, non_blocking=True).float().div_(255.0)   # uint8 -> [0,1] on GPU
    y = y.to(device, non_blocking=True)
    return x, y


In [ ]:
# ================== 200³ data, streamed from Hugging Face ==================
# harvardairobotics/Harvard-GF  (3,300 scans; 2100 / 300 / 900 train/val/test)
HF_REPO  = "harvardairobotics/Harvard-GF"
ZIP_FILE = "Dataset/dataset.zip"       # per-scan .npz with key 'oct_bscans' (200³ uint8)
CSV_FILE = "ReadMe/data_summary.csv"   # columns: filename,glaucoma(yes/no),use(training/validation/test)
DATA_DIR = "/content/glaucoma_hf_200"  # consolidated .npy arrays land here (cached on disk)
SPLITS   = ("Training", "Validation", "Test")

RESOLUTION  = 128                      # store arrays at this size (resize 200->R if R != 200);
                                       # pick a multiple of 32 for SwinUNETR (e.g. 128/192)
BATCH_SIZE   = 2                       # 200³ -> tiny per-step batch
GRAD_ACCUM   = 8                       # effective batch = 2 * 8 = 16
CACHE_IN_RAM = False                   # 200³ = ~26 GB total -> mmap, never hold in RAM
NUM_WORKERS  = max(2, os.cpu_count() or 2)

SPLIT_ALIAS = {"training": "Training", "validation": "Validation", "valid": "Validation",
               "test": "Test", "testing": "Test"}


def download_hf(filename):
    from huggingface_hub import hf_hub_download
    print(f"[data] downloading {HF_REPO}/{filename} ...", flush=True)
    return hf_hub_download(repo_id=HF_REPO, filename=filename, repo_type="dataset")


def build_200_data():
    """Stream Harvard-GF -> per-split 200³ .npy arrays (once, low RAM, disk cached)."""
    if all(os.path.isfile(os.path.join(DATA_DIR, f"{s}_volumes.npy")) for s in SPLITS):
        print(f"[data] already built at {DATA_DIR}")
        return
    os.makedirs(DATA_DIR, exist_ok=True)
    csv_path = download_hf(CSV_FILE)
    zip_path = download_hf(ZIP_FILE)

    import csv
    meta = {}
    with open(csv_path, newline="") as fh:
        for r in csv.DictReader(fh):
            split = SPLIT_ALIAS.get((r["use"] or "").strip().lower())
            if split is None:
                continue
            gl = 1 if str(r["glaucoma"]).strip().lower() in ("yes", "1", "true") else 0
            meta[Path(r["filename"]).stem] = (split, gl)
    print(f"[data] {len(meta)} labeled samples from CSV")

    with zipfile.ZipFile(zip_path) as zf:
        names = [n for n in zf.namelist() if n.endswith(".npz")]
        counts = {s: 0 for s in SPLITS}
        for n in names:
            m = meta.get(Path(n).stem)
            if m:
                counts[m[0]] += 1
    print("[data] zip-matched counts:", counts)

    vols, labels = {}, {}
    for s in SPLITS:
        vp = os.path.join(DATA_DIR, f"{s}_volumes.npy")
        vols[s] = np.lib.format.open_memmap(vp, mode="w+", dtype=np.uint8,
                                            shape=(counts[s], 1, RESOLUTION, RESOLUTION, RESOLUTION))
        labels[s] = np.zeros(counts[s], dtype=np.int64)

    filled = {s: 0 for s in SPLITS}
    with zipfile.ZipFile(zip_path) as zf:
        for n in names:
            m = meta.get(Path(n).stem)
            if not m:
                continue
            split, label = m
            raw = np.load(io.BytesIO(zf.read(n)))["oct_bscans"]      # (200,200,200) uint8
            if RESOLUTION != 200:
                t = torch.from_numpy(raw).float().div_(255.0).unsqueeze(0).unsqueeze(0)
                t = torch.nn.functional.interpolate(
                    t, size=(RESOLUTION,) * 3, mode="trilinear", align_corners=False)
                raw = (t.squeeze(0, 1).clamp(0, 1) * 255).round().numpy().astype(np.uint8)
            vols[split][filled[split]] = raw[None]                   # -> (1,R,R,R)
            labels[split][filled[split]] = label
            filled[split] += 1
    for s in SPLITS:
        vols[s].flush()
        np.save(os.path.join(DATA_DIR, f"{s}_labels.npy"), labels[s])
        print(f"[data] {s}: {filled[s]} volumes ({(counts[s] * 8 / 1e9):.1f} GB)")
    with open(os.path.join(DATA_DIR, "manifest.json"), "w") as fh:
        json.dump({"source": HF_REPO, "size_name": "200", "store_shape": [1, 200, 200, 200],
                   "splits": {s: {"built_n": filled[s]} for s in SPLITS}}, fh, indent=2)


class OCTMemmapDataset(Dataset):
    """Consolidated {split}_volumes.npy is (N,1,200,200,200) uint8; labels (N,) int64."""

    def __init__(self, data_dir, split, cache_in_ram=False):
        self.labels = np.load(os.path.join(data_dir, f"{split}_labels.npy"))
        vp = os.path.join(data_dir, f"{split}_volumes.npy")
        self.volumes = np.load(vp) if cache_in_ram else np.load(vp, mmap_mode="r")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.from_numpy(np.ascontiguousarray(self.volumes[idx]).copy())  # uint8 (1,200,200,200), writable
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y


def build_loaders():
    build_200_data()
    counts = {s: len(np.load(os.path.join(DATA_DIR, f"{s}_labels.npy"))) for s in SPLITS}
    print("[data] counts:", counts)
    workers = 0 if CACHE_IN_RAM else NUM_WORKERS
    kw = dict(batch_size=BATCH_SIZE, num_workers=workers, pin_memory=(device == "cuda"))
    if workers > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    train_ds = OCTMemmapDataset(DATA_DIR, "Training",   cache_in_ram=CACHE_IN_RAM)
    val_ds   = OCTMemmapDataset(DATA_DIR, "Validation", cache_in_ram=CACHE_IN_RAM)
    test_ds  = OCTMemmapDataset(DATA_DIR, "Test",       cache_in_ram=CACHE_IN_RAM)
    g = torch.Generator(); g.manual_seed(SEED)
    train_loader = DataLoader(train_ds, shuffle=True, generator=g, **kw)
    val_loader   = DataLoader(val_ds,   shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, **kw)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_loaders()


In [ ]:
# ---- conservative 3D augmentation (training only; keeps OCT structure) ----
def make_train_transform():
    return Compose([
        RandFlip(prob=0.5, spatial_axis=1),                    # flip H
        RandFlip(prob=0.5, spatial_axis=2),                    # flip W
        RandRotate(range_x=0.10, range_y=0.10, range_z=0.10,   # ±~6°
                   prob=0.5, mode="bilinear", padding_mode="zeros", keep_size=True),
        RandScaleIntensity(factors=0.10, prob=0.5),            # *(1±0.1)
        RandShiftIntensity(offsets=10.0, prob=0.5),            # ±10 on [0,255]
        RandGaussianNoise(prob=0.3, std=5.0),
    ])


class TrainAug:
    """Wraps the memmap train split and applies MONAI transforms on-the-fly."""

    def __init__(self, ds, transform):
        self.ds, self.tf = ds, transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        x, y = self.ds[i]
        x = torch.as_tensor(self.tf(x)[0])      # MONAI may drop the singleton channel
        if x.ndim == 3:
            x = x.unsqueeze(0)                  # restore (1,200,200,200)
        return x, y


g = torch.Generator(); g.manual_seed(SEED)
workers = 0 if CACHE_IN_RAM else NUM_WORKERS
train_loader = DataLoader(TrainAug(train_loader.dataset, make_train_transform()),
                          batch_size=BATCH_SIZE, shuffle=True, generator=g,
                          num_workers=workers, pin_memory=(device == "cuda"),
                          persistent_workers=workers > 0, prefetch_factor=4 if workers > 0 else None)


In [ ]:
class UNetEncoder3D(nn.Module):
    """MONAI UNet ENCODER ONLY (decoder removed) + AdaptiveAvgPool + classification head."""

    def __init__(self, in_channels=1, num_classes=2, features=(32, 64, 128, 256),
                 strides=(2, 2, 2), num_res_units=2, norm="batch", dropout=0.0):
        super().__init__()
        self.features = tuple(features)
        strides = tuple(strides)
        if len(strides) != len(features) - 1:      # one pool between consecutive levels
            strides = (2,) * (len(features) - 1)
        self.down_blocks = nn.ModuleList()
        self.down_samples = nn.ModuleList()
        cin = in_channels
        for i, f in enumerate(features):
            self.down_blocks.append(UnetBasicBlock(
                3, cin, f, kernel_size=3, stride=1,
                norm_name=norm, act_name="relu", dropout=dropout))
            cin = f
            if i < len(features) - 1:
                self.down_samples.append(nn.MaxPool3d(strides[i], stride=strides[i]))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(features[-1], num_classes),
        )

    def forward(self, x):
        for blk, samp in zip(self.down_blocks, self.down_samples + [nn.Identity()]):
            x = blk(x)
            x = samp(x)
        return self.head(x)


class Simple3DCNN(nn.Module):
    """Project reference baseline (models/glaucoma/model.py, unchanged)."""

    def __init__(self, in_channels=1, num_classes=2, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(in_channels, 16, kernel_size=3, padding=1), nn.BatchNorm3d(16),
            nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(16, 32, kernel_size=3, padding=1), nn.BatchNorm3d(32),
            nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(32, 64, kernel_size=3, padding=1), nn.BatchNorm3d(64),
            nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(64, 128, kernel_size=3, padding=1), nn.BatchNorm3d(128),
            nn.ReLU(), nn.AdaptiveAvgPool3d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


@torch.no_grad()
def eval_acc(model, loader):
    model.eval(); c = t = 0
    for x, y in loader:
        x, y = to_device_normalize(x, y)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            logits = model(x)
        c += (logits.argmax(1) == y).sum().item(); t += y.numel()
    return c / t


In [ ]:
# ====== 3D segmentation backbones -> encoder + classification head ======
# Each wrapper keeps only the ENCODER of a well-known 3D segmentation model and
# returns a global-pooled (B, out_dim) embedding that SegEncHead classifies.

class SegEncHead(nn.Module):
    """Segmentation-encoder + classification head."""

    def __init__(self, backbone, num_classes=2, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(backbone.out_dim, num_classes))

    def forward(self, x):
        return self.head(self.backbone(x))


class MONAIDynUNetEnc(nn.Module):          # nnU-Net v1 / ResEnc-style (MONAI DynUNet)
    def __init__(self, in_channels=1, filters=(16, 32, 64, 128, 256), res_block=True):
        super().__init__()
        self.net = nets.DynUNet(spatial_dims=3, in_channels=in_channels, out_channels=1,
                                kernel_size=[3] * len(filters),
                                strides=[1] + [2] * (len(filters) - 1),
                                upsample_kernel_size=[2] * len(filters),
                                filters=list(filters),
                                norm_name="batch", act_name=("relu", {"inplace": True}),
                                res_block=res_block)
        self.out_dim = filters[-1]

    def forward(self, x):
        x = self.net.input_block(x)
        for d in self.net.downsamples:
            x = d(x)
        return F.adaptive_avg_pool3d(self.net.bottleneck(x), 1).flatten(1)


class MONAIVNetEnc(nn.Module):             # V-Net
    def __init__(self, in_channels=1):
        super().__init__()
        self.net = nets.VNet(spatial_dims=3, in_channels=in_channels, out_channels=1)
        self.out_dim = 256

    def forward(self, x):
        o16 = self.net.in_tr(x)
        o32 = self.net.down_tr32(o16)
        o64 = self.net.down_tr64(o32)
        o128 = self.net.down_tr128(o64)
        o256 = self.net.down_tr256(o128)
        return F.adaptive_avg_pool3d(o256, 1).flatten(1)


class MONAISegResEnc(nn.Module):           # SegResNet (3D MRI segmentation)
    def __init__(self, in_channels=1, init_filters=16):
        super().__init__()
        self.net = nets.SegResNet(spatial_dims=3, in_channels=in_channels, out_channels=1,
                                  init_filters=init_filters)
        self.out_dim = init_filters * 2 ** (len(self.net.blocks_down) - 1)

    def forward(self, x):
        x, _ = self.net.encode(x)          # encoder = convInit + down_layers
        return F.adaptive_avg_pool3d(x, 1).flatten(1)


class MONAIUNETREnc(nn.Module):            # UNETR (ViT encoder)
    def __init__(self, in_channels=1, img_size=(RESOLUTION, RESOLUTION, RESOLUTION),
                 hidden_size=384, num_heads=12, mlp_dim=2048):
        super().__init__()
        self.net = nets.UNETR(in_channels=in_channels, out_channels=1, img_size=img_size,
                              feature_size=16, hidden_size=hidden_size, mlp_dim=mlp_dim,
                              num_heads=num_heads, proj_type="conv", norm_name="instance")
        self.out_dim = hidden_size

    def forward(self, x):
        tokens, _ = self.net.vit(x)        # (B, n_patches+1, hidden)
        return tokens.mean(dim=1)          # global mean-pool of patch tokens


class MONAISwinUNETREnc(nn.Module):        # SwinUNETR (Swin-transformer encoder)
    def __init__(self, in_channels=1, feature_size=24, depths=(2, 2, 2, 2),
                 num_heads=(3, 6, 12, 24), window_size=7):
        super().__init__()
        self.net = nets.SwinUNETR(in_channels=in_channels, out_channels=1, feature_size=feature_size,
                                  depths=depths, num_heads=num_heads, window_size=window_size,
                                  spatial_dims=3)
        self.out_dim = feature_size * 2 ** len(depths)

    def forward(self, x):
        hs = self.net.swinViT(x, True)     # list of stage feature maps
        return F.adaptive_avg_pool3d(hs[-1], 1).flatten(1)


# ----- official nnU-Net v2 encoders (dynamic_network_architectures) -----
try:
    from dynamic_network_architectures.architectures.unet import (
        ResidualEncoder, PlainConvEncoder)
    NNUNET_AVAILABLE = True
except Exception:
    ResidualEncoder = PlainConvEncoder = None
    NNUNET_AVAILABLE = False
    print("WARNING: nnunetv2 not importable — nnU-Net v2 rows will be skipped.")


class NNUNetV2Enc(nn.Module):              # official nnU-Net v2 encoder (PlainConv / ResEnc)
    def __init__(self, kind="plain", in_channels=1, features=(32, 64, 128, 256, 320, 320),
                 strides=(1, 2, 2, 2, 2, 2), n_per_stage=(2, 2, 2, 2, 2, 2)):
        super().__init__()
        kw = dict(input_channels=in_channels, n_stages=len(features),
                  features_per_stage=list(features), conv_op=nn.Conv3d,
                  kernel_sizes=3, strides=list(strides), conv_bias=True,
                  norm_op=nn.InstanceNorm3d, norm_op_kwargs={"eps": 1e-5, "affine": True},
                  dropout_op=None, dropout_op_kwargs=None,
                  nonlin=nn.LeakyReLU, nonlin_kwargs={"inplace": True})
        self.encoder = (ResidualEncoder(n_blocks_per_stage=list(n_per_stage), **kw)
                        if kind == "residual"
                        else PlainConvEncoder(n_conv_per_stage=list(n_per_stage), **kw))
        self.out_dim = features[-1]

    def forward(self, x):
        return F.adaptive_avg_pool3d(self.encoder(x), 1).flatten(1)


def build_model(spec):
    """Build a classification model (forward -> logits) from a backbone spec dict."""
    kind = spec["kind"]
    if kind == "unetenc":
        return UNetEncoder3D(in_channels=1, num_classes=2, features=(32, 64, 128, 256),
                             num_res_units=2)
    if kind == "dynunet":
        bb = MONAIDynUNetEnc(filters=spec.get("filters", (16, 32, 64, 128, 256)),
                             res_block=spec.get("res_block", True))
    elif kind == "vnet":
        bb = MONAIVNetEnc()
    elif kind == "segresnet":
        bb = MONAISegResEnc(init_filters=spec.get("init_filters", 16))
    elif kind == "unetr":
        bb = MONAIUNETREnc(hidden_size=spec.get("hidden_size", 384))
    elif kind == "swinunetr":
        bb = MONAISwinUNETREnc(feature_size=spec.get("feature_size", 24))
    elif kind == "nnunetv2":
        if not NNUNET_AVAILABLE:
            raise RuntimeError("nnunetv2 not installed — run the setup cell / pip install nnunetv2")
        bb = NNUNetV2Enc(kind=spec.get("v2", "plain"),
                         features=spec.get("features", (32, 64, 128, 256, 320, 320)))
    else:
        raise ValueError(f"unknown backbone kind: {kind}")
    return SegEncHead(bb, num_classes=2, dropout=0.3)


In [ ]:
# ====== sweep over 3D segmentation-encoder backbones (A100 80GB, resolution from data cell) ======
SWEEP = [
    {"family": "MONAI-SegBackbone", "kind": "dynunet",    "name": "dynunet-16",  "filters": (16, 32, 64, 128, 256)},
    {"family": "MONAI-SegBackbone", "kind": "dynunet",    "name": "dynunet-32",  "filters": (32, 64, 128, 256, 320)},
    {"family": "MONAI-SegBackbone", "kind": "vnet",       "name": "vnet"},
    {"family": "MONAI-SegBackbone", "kind": "segresnet",  "name": "segresnet-16"},
    {"family": "MONAI-Transformer", "kind": "unetr",      "name": "unetr-h384"},
    {"family": "MONAI-Transformer", "kind": "swinunetr",  "name": "swinunetr-f24"},
    {"family": "nnU-Net-v2",        "kind": "nnunetv2",   "name": "nnunetv2-plain", "v2": "plain"},
    {"family": "nnU-Net-v2",        "kind": "nnunetv2",   "name": "nnunetv2-res",   "v2": "residual"},
    {"family": "Baseline",          "kind": "unetenc",    "name": "unetenc-32"},
]
SWEEP_EPOCHS = 20
EFFECTIVE_BS = BATCH_SIZE * GRAD_ACCUM
LR  = 1e-4 * (EFFECTIVE_BS / 4) ** 0.5
WD, PATIENCE = 1e-4, 10


def _probe_loader(bs):
    return DataLoader(train_loader.dataset, batch_size=bs, shuffle=True,
                      num_workers=0, generator=torch.Generator().manual_seed(SEED))


def static_probe(spec, bs=None):
    """Probe fit at batch `bs` (default BATCH_SIZE); on failure retry at batch 1.
    Returns (params_M, vram_GB, status, batch_used)."""
    bs = BATCH_SIZE if bs is None else bs
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    try:
        m = build_model(spec).to(device)
        opt = torch.optim.AdamW(m.parameters(), LR, weight_decay=WD)
        xb, yb = next(iter(_probe_loader(bs))); xb, yb = to_device_normalize(xb, yb)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            loss = nn.functional.cross_entropy(m(xb), yb)
        loss.backward(); opt.step()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        params = sum(p.numel() for p in m.parameters()) / 1e6
        vram = (torch.cuda.max_memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
        del m, opt, xb, yb, loss; torch.cuda.empty_cache()
        return params, vram, ("ok" if bs == BATCH_SIZE else "ok@bs1"), bs
    except Exception as e:
        torch.cuda.empty_cache()
        if bs > 1:
            print(f"  [probe] bs={bs} failed ({type(e).__name__}: {str(e)[:160]}); retrying bs=1")
            return static_probe(spec, bs=1)
        st = "OOM" if "out of memory" in str(e).lower() else "ERR"
        print(f"  [probe] {st}: {type(e).__name__}: {str(e)[:300]}")
        return float("nan"), float("nan"), st, 1


def train_one(spec, bs=BATCH_SIZE):
    torch.manual_seed(SEED); torch.cuda.empty_cache()
    name = spec["name"]
    try:
        model = build_model(spec).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=SWEEP_EPOCHS)
        scaler = torch.amp.GradScaler("cuda",
                    enabled=(USE_AMP and device == "cuda" and amp_dtype == torch.float16))
        crit = nn.CrossEntropyLoss()
        loader = train_loader if bs == BATCH_SIZE else _probe_loader(bs)
        best_val = best_test = 0.0; best_ep = -1; bad = 0; ep_times = []
        for ep in range(SWEEP_EPOCHS):
            model.train(); t0 = time.time(); opt.zero_grad(set_to_none=True)
            for i, (x, y) in enumerate(loader):
                x, y = to_device_normalize(x, y)
                with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
                    loss = crit(model(x), y) / GRAD_ACCUM
                scaler.scale(loss).backward()
                if (i + 1) % GRAD_ACCUM == 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            sched.step(); ep_times.append(time.time() - t0)
            va = eval_acc(model, val_loader)
            if va > best_val:
                best_val, best_ep = va, ep; best_test = eval_acc(model, test_loader); bad = 0
            else:
                bad += 1
                if bad >= PATIENCE:
                    break
            print(f"  [{name}] ep{ep+1:02d} val={va:.4f} (best {best_val:.4f})", end="\r")
        del model, opt; torch.cuda.empty_cache()
        return dict(val=best_val, test=best_test, best_ep=best_ep + 1,
                    sec_ep=sum(ep_times) / len(ep_times), status="ok")
    except RuntimeError as e:
        torch.cuda.empty_cache()
        st = "OOM" if "out of memory" in str(e).lower() else "ERR"
        print(f"  [train] {st}: {type(e).__name__}: {str(e)[:300]}")
        return dict(val=float("nan"), test=float("nan"), best_ep=-1, sec_ep=float("nan"), status=st)


# ====== run (resume-safe: saves after each model, skips already-done ones) ======
RESULTS_LOCAL = "/content/sweep_results_seg_backbones.json"
RESULTS_DRIVE = "/content/drive/MyDrive/MasterBKDN/Thesis/sweep_results_seg_backbones.json"


def load_existing():
    for p in (RESULTS_LOCAL, RESULTS_DRIVE):
        try:
            with open(p) as fh:
                return json.load(fh)
        except Exception:
            continue
    return []


def save_rows(rows):
    try:
        with open(RESULTS_LOCAL, "w") as fh:
            json.dump(rows, fh, indent=2)
        os.makedirs(os.path.dirname(RESULTS_DRIVE), exist_ok=True)
        import shutil
        shutil.copy(RESULTS_LOCAL, RESULTS_DRIVE)
        print(f"  [save] {len(rows)} results -> {RESULTS_DRIVE}", flush=True)
    except Exception as e:
        print("  [save] skipped:", e, flush=True)


rows = load_existing()
done = {r["name"] for r in rows}

for spec in SWEEP:
    name = spec["name"]
    print(f"\n=== [{spec['family']}] {name} ===")
    if name in done:
        print(f"  already done -> skipping (resume); see row below")
        continue
    params, vram, st, bs = static_probe(spec)
    if st not in ("ok", "ok@bs1"):
        print(f"  static probe: {st}")
        rows.append(dict(family=spec["family"], name=name, params=params, vram=vram,
                         val=float("nan"), test=float("nan"), best_ep=-1,
                         sec_ep=float("nan"), status=st, bs=bs))
        save_rows(rows)
        continue
    print(f"  params {params:.1f}M | VRAM {vram:.1f}GB | batch={bs}")
    r = train_one(spec, bs=bs)
    r.update(family=spec["family"], name=name, params=params, vram=vram, bs=bs)
    rows.append(r)
    save_rows(rows)
    print(f"  val={r['val']:.4f} test={r['test']:.4f} @ep{r['best_ep']} | {r['sec_ep']:.1f}s/ep   ")

# ====== table ======
def fmt(r):
    return (f"{r['name']:18s} {r['params']:>6.1f}M {r['vram']:>5.1f}G {r['sec_ep']:>5.1f}s "
            f"{r['val']:>7.4f} {r['test']:>7.4f} {r['best_ep']:>6d}  b{r.get('bs', BATCH_SIZE)}  {r['status']}")

hdr = f"{'model':18s} {'params':>7} {'VRAM':>6} {'s/ep':>6} {'Val':>7} {'Test':>7} {'bestEp':>6} {'bs':>2}"
print(f"\n================= SWEEP {RESOLUTION}³ (3D SEGMENTATION ENCODERS + head) =================")
for fam in sorted({r["family"] for r in rows}):
    print(f"\n-- {fam} --\n{hdr}")
    for r in rows:
        if r["family"] == fam:
            print(fmt(r))

ok = sorted([r for r in rows if r["status"] == "ok"], key=lambda r: -r["val"])
print("\n----- TOP theo Val -----\n" + hdr)
for r in ok[:5]:
    print(fmt(r))
if ok:
    w = ok[0]
    WINNER = w
    print(f"\n>> WINNER: {w['name']} ({w['family']}) Val={w['val']:.4f} Test={w['test']:.4f} "
          f"| {w['params']:.1f}M | {w['vram']:.1f}GB | batch={w['bs']}")
else:
    WINNER = None

# results are already saved incrementally above; just confirm on Drive
print(f"[done] {len(rows)} rows (resume file: {RESULTS_DRIVE})")


In [ ]:
# Done. Release the GPU immediately.
from google.colab import runtime
runtime.unassign()
